# Etapa 1: curadoria

Entrada: `data/raw/loan_approval_dataset.csv`, baixado do Kaggle (*Loan-Approval-Prediction-Dataset*, versão 1, de 16/07/2023, licença MIT). A página não informa a origem dos dados, o significado das colunas nem a unidade monetária. O score de crédito é o CIBIL, birô de crédito da Índia (escala de 300 a 900), o que sugere valores em rúpias.

Saída: `data/processed/emprestimos.csv`, com nomes e valores limpos, categorias em 0/1 e os patrimônios negativos corrigidos.

## 0. Configuração

In [1]:
import sys
import hashlib
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config

sha256 = hashlib.sha256(config.RAW_CSV.read_bytes()).hexdigest().upper()
print("SHA-256 confere:", sha256 == config.RAW_SHA256)

bruto = pd.read_csv(config.RAW_CSV)
print(bruto.shape)
bruto.head()

SHA-256 confere: True
(4269, 13)


,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


## 1. Nomes e valores com espaços

Os nomes das colunas e os valores de texto do CSV começam com um espaço.

In [2]:
print([repr(c) for c in bruto.columns])
print(bruto[" loan_status"].unique())

["'loan_id'", "' no_of_dependents'", "' education'", "' self_employed'", "' income_annum'", "' loan_amount'", "' loan_term'", "' cibil_score'", "' residential_assets_value'", "' commercial_assets_value'", "' luxury_assets_value'", "' bank_asset_value'", "' loan_status'"]
<StringArray>
[' Approved', ' Rejected']
Length: 2, dtype: str


`.str.strip()` remove os espaços do começo e do fim. A cópia mantém `bruto` intacto para comparação.

In [3]:
base = bruto.copy()
base.columns = base.columns.str.strip()
for coluna in ["education", "self_employed", "loan_status"]:
    base[coluna] = base[coluna].str.strip()

print(list(base.columns))
print(base["loan_status"].unique())

['loan_id', 'no_of_dependents', 'education', 'self_employed', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value', 'loan_status']
<StringArray>
['Approved', 'Rejected']
Length: 2, dtype: str


## 2. Verificações de qualidade

In [4]:
print("valores faltantes:", int(base.isna().sum().sum()))
print("linhas duplicadas:", int(base.duplicated().sum()))
print("loan_id único:", base[config.ID].is_unique)
for coluna in ["education", "self_employed", "loan_status"]:
    print(base[coluna].value_counts(), "\n")

valores faltantes: 0
linhas duplicadas: 0
loan_id único: True
education
Graduate        2144
Not Graduate    2125
Name: count, dtype: int64 

self_employed
Yes    2150
No     2119
Name: count, dtype: int64 

loan_status
Approved    2656
Rejected    1613
Name: count, dtype: int64 



Não há valores faltantes nem linhas duplicadas, e cada empréstimo tem um `loan_id` próprio. Educação e trabalho autônomo dividem a base quase ao meio.

## 3. Patrimônio residencial negativo

In [5]:
negativos = base[base["residential_assets_value"] < 0]
print("linhas:", len(negativos))
print("valores distintos:", negativos["residential_assets_value"].unique())
print("aprovação nessas linhas:", round((negativos["loan_status"] == "Approved").mean(), 3))
print("aprovação geral:", round((base["loan_status"] == "Approved").mean(), 3))
print("linhas com valor 0:", int((base["residential_assets_value"] == 0).sum()))

linhas: 28
valores distintos: [-100000]
aprovação nessas linhas: 0.571
aprovação geral: 0.622
linhas com valor 0: 45


As 28 linhas negativas têm o mesmo valor, −100.000. Todos os patrimônios da base variam em passos de 100.000 a partir de 0, então o valor parece um passo abaixo de zero gerado por engano. Patrimônio negativo não tem interpretação; o valor é substituído por 0 ("sem patrimônio residencial"). O efeito é desprezível perto do desvio da coluna (cerca de 6,5 milhões), e nenhuma linha é descartada.

As três colunas de texto viram 0/1, porque a rede neural só trabalha com números: `education` = 1 para Graduate, `self_employed` = 1 para Yes e `loan_status` = 1 para Approved.

In [6]:
base.loc[base["residential_assets_value"] < 0, "residential_assets_value"] = 0

base["education"] = (base["education"] == "Graduate").astype(int)
base["self_employed"] = (base["self_employed"] == "Yes").astype(int)
base["loan_status"] = (base["loan_status"] == "Approved").astype(int)

base.head()

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,1,0,9600000,29900000,12,778,2400000,17600000,22700000,8000000,1
1,2,0,0,1,4100000,12200000,8,417,2700000,2200000,8800000,3300000,0
2,3,3,1,0,9100000,29700000,20,506,7100000,4500000,33300000,12800000,0
3,4,3,1,0,8200000,30700000,8,467,18200000,3300000,23300000,7900000,0
4,5,5,0,1,9800000,24200000,20,382,12400000,8200000,29400000,5000000,0


## 4. Base final

In [7]:
assert list(base.columns) == [config.ID] + config.FEATURES + [config.TARGET]
assert (base[config.FEATURES] >= 0).all().all()
assert base[config.TARGET].isin([0, 1]).all()

config.PROCESSED_CSV.parent.mkdir(parents=True, exist_ok=True)
base.to_csv(config.PROCESSED_CSV, index=False)
print("gravado:", config.PROCESSED_CSV.name, base.shape)
print("aprovados:", round(base[config.TARGET].mean(), 3))

gravado: emprestimos.csv (4269, 13)
aprovados: 0.622


**Resultado.** 4.269 empréstimos e 13 colunas, sem valores faltantes nem linhas duplicadas. Foram removidos os espaços dos nomes e dos valores, as três colunas de texto viraram 0/1 e os 28 patrimônios residenciais de −100.000 viraram 0. O alvo tem 62,2% de aprovados. A coluna `loan_id` fica na base para identificar as linhas, mas não entra como atributo.